In [ ]:
# Step 1: Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.svm import SVC

# Step 2: Synthetic dataset (1000 transactions)
np.random.seed(42)
amount = np.random.exponential(scale=100, size=1000)   # transaction amount
time = np.random.randint(0, 24, 1000)                  # transaction hour
location = np.random.randint(1, 5, 1000)               # location code
fraud = ((amount > 200) & (time < 6)).astype(int)      # fraud rule

data = pd.DataFrame({
    'amount': amount,
    'time': time,
    'location': location,
    'fraud': fraud
})

X = data[['amount','time','location']]
y = data['fraud']

# Step 3: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Step 4: Handle imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Step 5: Train XGBoost
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train_res, y_train_res)

# Step 6: Predictions
y_pred_prob = xgb.predict_proba(X_test)[:,1]

# Threshold tuning (e.g. 0.3 instead of 0.5)
threshold = 0.3
y_pred = (y_pred_prob >= threshold).astype(int)

# Step 7: Evaluation
print("XGBoost ROC-AUC:", roc_auc_score(y_test, y_pred_prob))
print(classification_report(y_test, y_pred))

# Step 8: Feature importance
import matplotlib.pyplot as plt
import xgboost as xgb_lib
xgb.plot_importance(xgb)
plt.show()

# Step 9: Baseline SVM
svm = SVC(probability=True)
svm.fit(X_train_res, y_train_res)
y_pred_svm_prob = svm.predict_proba(X_test)[:,1]
print("SVM ROC-AUC:", roc_auc_score(y_test, y_pred_svm_prob))
